In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("..")

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
import tqdm

from plots import hist_events_by_labels
from events_data import EventsData
from fvt_classifier import FvTClassifier
# import LogNorm
from matplotlib.colors import LogNorm
from training_info import TrainingInfo
from plots import plot_rewighted_samples_by_model
from dataset import MotherSamples
from events_data import events_from_scdinfo
import pickle
import pandas as pd


features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

# use tex
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"

plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.titlesize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["figure.labelsize"] = 20
plt.rcParams["lines.markersize"] = 3

In [4]:
import pickle
from signal_region import get_SR_CR_cut, get_DRs
from utils import get_quantiles_with_weights
import pandas as pd
from plots import plot_sr_stats
from ancillary_features import get_closest_dijet_masses

In [14]:

n_3b = 100_000
device = torch.device("cuda")
experiment_name = "smeared_fvt_training_small"
signal_filename = "HH4b_picoAOD.h5"
ratio_4b = 0.5
seeds = np.arange(10)

df_list = []
    
for signal_ratio in [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]:
    for seed in tqdm.tqdm(seeds):
        hparam_filter = {
            "experiment_name": experiment_name,
            "dataset": lambda x: all([x["seed"] == seed, 
                                x["n_3b"] == n_3b, 
                                x["signal_ratio"] == signal_ratio]),
            "aux_info_step": 2, 
        }
        hashes = TrainingInfo.find(hparam_filter)
        assert len(hashes) == 3, f"Expected 3 hashes for seed {seed}, got {len(hashes)}"
        
        smeared_fvt_tinfo = TrainingInfo.load(hashes[0])
        msamples = MotherSamples.load(smeared_fvt_tinfo.ms_hash)
        tst_scdinfo = msamples.scdinfo[~smeared_fvt_tinfo.ms_idx]
        events_tst = events_from_scdinfo(tst_scdinfo, features, signal_filename)

        for hash_idx, smeared_fvt_hash in enumerate(hashes):
            smeared_fvt_tinfo = TrainingInfo.load(smeared_fvt_hash)
            smeared_fvt_model = smeared_fvt_tinfo.load_trained_model(
                    smeared_fvt_tinfo.hparams["encoder_mode"]
                )
            smeared_fvt_model.eval()
            smeared_fvt_model.to(torch.device("cuda"))
            noise_scale = smeared_fvt_tinfo.hparams["smearing"]["noise_scale"]
            
            if hash_idx == 0:
                # calculate gamma_base
                base_encoder_hash = smeared_fvt_tinfo.hparams["encoder_hash"]

                base_fvt_model = TrainingInfo.load(base_encoder_hash).load_trained_model(
                    smeared_fvt_tinfo.hparams["encoder_mode"]
                )
                base_fvt_model.eval()
                base_fvt_model.to(torch.device("cuda"))

                # Use the same mother samples and exclude ones used for training base & smeared FvT model
                gamma_base, gamma_smeared, q_repr_base = get_DRs(events_tst, base_fvt_model, smeared_fvt_model, return_repr=True)    
                            
            else:
                # only need to compute gamma_smeared
                fvt_smeared = smeared_fvt_model.predict(q_repr_base)[:, 1].detach().cpu().numpy()
                gamma_smeared = fvt_smeared / (1 - fvt_smeared)
            
            SR_stats = np.log(gamma_base / gamma_smeared)
            SR_stats_4b = SR_stats[events_tst.is_4b]
            SR_stats_signal = SR_stats[events_tst.is_signal]
            weights_4b = events_tst.weights[events_tst.is_4b]
            weights_signal = events_tst.weights[events_tst.is_signal]
            w_signal_sum = np.sum(weights_signal)
            
            w_4b_cut = np.arange(0, 1.05, 0.05)
            SR_stats_thrs = get_quantiles_with_weights(SR_stats_4b, weights_4b, w_4b_cut[::-1])
            w_signal = [np.sum(weights_signal[SR_stats_signal > SR_stats_thrs[i]]) / w_signal_sum for i in range(len(SR_stats_thrs))]
            
            for i in range(len(SR_stats_thrs)):
                df_list.append({
                    "w_signal": w_signal[i],
                    "w_4b_cut": w_4b_cut[i],
                    "seed": seed,
                    "signal_ratio": signal_ratio,
                    "n_3b": n_3b,
                    "noise_scale": noise_scale,
                })
                
        df = pd.DataFrame(df_list)
        df.to_csv("./data/tsv/SR_stats_vs_w_4b.csv", index=False)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:39<00:00,  3.98s/it]


In [15]:
n_3b = 100_0000
device = torch.device("cuda")
experiment_name = "smeared_fvt_training"
signal_filename = "HH4b_picoAOD.h5"
ratio_4b = 0.5
seeds = np.arange(10)

df_list = []
    
for signal_ratio in [0.01, 0.02]:
    for seed in tqdm.tqdm(seeds):
        hparam_filter = {
            "experiment_name": experiment_name,
            "dataset": lambda x: all([x["seed"] == seed, 
                                x["n_3b"] == n_3b, 
                                x["signal_ratio"] == signal_ratio]),
            "aux_info_step": 2, 
        }
        hashes = TrainingInfo.find(hparam_filter)
        # assert len(hashes) == 3, f"Expected 3 hashes for seed {seed}, got {len(hashes)}"
        if len(hashes) == 0:
            continue
        
        smeared_fvt_tinfo = TrainingInfo.load(hashes[0])
        msamples = MotherSamples.load(smeared_fvt_tinfo.ms_hash)
        tst_scdinfo = msamples.scdinfo[~smeared_fvt_tinfo.ms_idx]
        events_tst = events_from_scdinfo(tst_scdinfo, features, signal_filename)

        for hash_idx, smeared_fvt_hash in enumerate(hashes):
            smeared_fvt_tinfo = TrainingInfo.load(smeared_fvt_hash)
            smeared_fvt_model = smeared_fvt_tinfo.load_trained_model(
                    smeared_fvt_tinfo.hparams["encoder_mode"]
                )
            smeared_fvt_model.eval()
            smeared_fvt_model.to(torch.device("cuda"))
            noise_scale = smeared_fvt_tinfo.hparams["smearing"]["noise_scale"]
            
            if hash_idx == 0:
                # calculate gamma_base
                base_encoder_hash = smeared_fvt_tinfo.hparams["encoder_hash"]

                base_fvt_model = TrainingInfo.load(base_encoder_hash).load_trained_model(
                    smeared_fvt_tinfo.hparams["encoder_mode"]
                )
                base_fvt_model.eval()
                base_fvt_model.to(torch.device("cuda"))

                # Use the same mother samples and exclude ones used for training base & smeared FvT model
                gamma_base, gamma_smeared, q_repr_base = get_DRs(events_tst, base_fvt_model, smeared_fvt_model, return_repr=True)    
                            
            else:
                # only need to compute gamma_smeared
                fvt_smeared = smeared_fvt_model.predict(q_repr_base)[:, 1].detach().cpu().numpy()
                gamma_smeared = fvt_smeared / (1 - fvt_smeared)
            
            SR_stats = np.log(gamma_base / gamma_smeared)
            SR_stats_4b = SR_stats[events_tst.is_4b]
            SR_stats_signal = SR_stats[events_tst.is_signal]
            weights_4b = events_tst.weights[events_tst.is_4b]
            weights_signal = events_tst.weights[events_tst.is_signal]
            w_signal_sum = np.sum(weights_signal)
            
            w_4b_cut = np.arange(0, 1.05, 0.05)
            SR_stats_thrs = get_quantiles_with_weights(SR_stats_4b, weights_4b, w_4b_cut[::-1])
            w_signal = [np.sum(weights_signal[SR_stats_signal > SR_stats_thrs[i]]) / w_signal_sum for i in range(len(SR_stats_thrs))]
            
            for i in range(len(SR_stats_thrs)):
                df_list.append({
                    "w_signal": w_signal[i],
                    "w_4b_cut": w_4b_cut[i],
                    "seed": seed,
                    "signal_ratio": signal_ratio,
                    "n_3b": n_3b,
                    "noise_scale": noise_scale,
                })
                
df = pd.DataFrame(df_list)
df.to_csv("./data/tsv/SR_stats_vs_w_4b.csv", index=False, mode="a", header=False)

100%|██████████| 10/10 [01:33<00:00,  9.40s/it]


In [6]:
n_3b = 100_0000
device = torch.device("cuda")
experiment_name = "smeared_fvt_training_noise_scale"
signal_filename = "HH4b_picoAOD.h5"
ratio_4b = 0.5
seeds = np.arange(10)

df_list = []
    
for signal_ratio in [0.01, 0.02]:
    for seed in tqdm.tqdm(seeds):
        hparam_filter = {
            "experiment_name": experiment_name,
            "dataset": lambda x: all([x["seed"] == seed, 
                                x["n_3b"] == n_3b, 
                                x["signal_ratio"] == signal_ratio]),
            "aux_info_step": 2, 
        }
        hashes = TrainingInfo.find(hparam_filter)
        # assert len(hashes) == 3, f"Expected 3 hashes for seed {seed}, got {len(hashes)}"
        if len(hashes) == 0:
            continue
        
        smeared_fvt_tinfo = TrainingInfo.load(hashes[0])
        msamples = MotherSamples.load(smeared_fvt_tinfo.ms_hash)
        tst_scdinfo = msamples.scdinfo[~smeared_fvt_tinfo.ms_idx]
        events_tst = events_from_scdinfo(tst_scdinfo, features, signal_filename)
        gamma_base = None
        q_repr_base = None

        for hash_idx, smeared_fvt_hash in enumerate(hashes):
            smeared_fvt_tinfo = TrainingInfo.load(smeared_fvt_hash)
            smeared_fvt_model = smeared_fvt_tinfo.load_trained_model(
                    smeared_fvt_tinfo.hparams["encoder_mode"]
                )
            smeared_fvt_model.eval()
            smeared_fvt_model.to(torch.device("cuda"))
            noise_scale = smeared_fvt_tinfo.hparams["smearing"]["noise_scale"]
            
            if gamma_base is None:
                base_encoder_hash = smeared_fvt_tinfo.hparams["encoder_hash"]

                base_fvt_model = TrainingInfo.load(base_encoder_hash).load_trained_model(
                    smeared_fvt_tinfo.hparams["encoder_mode"]
                )
                base_fvt_model.eval()
                base_fvt_model.to(torch.device("cuda"))

                # Use the same mother samples and exclude ones used for training base & smeared FvT model
                gamma_base, gamma_smeared, q_repr_base = get_DRs(events_tst, base_fvt_model, smeared_fvt_model, return_repr=True)    
                
            else:
                # only need to compute gamma_smeared
                fvt_smeared = smeared_fvt_model.predict(q_repr_base)[:, 1].detach().cpu().numpy()
                gamma_smeared = fvt_smeared / (1 - fvt_smeared)
                
            
            SR_stats = np.log(gamma_base / gamma_smeared)
            SR_stats_4b = SR_stats[events_tst.is_4b]
            SR_stats_signal = SR_stats[events_tst.is_signal]
            weights_4b = events_tst.weights[events_tst.is_4b]
            weights_signal = events_tst.weights[events_tst.is_signal]
            w_signal_sum = np.sum(weights_signal)
            
            w_4b_cut = np.arange(0, 1.05, 0.05)
            SR_stats_thrs = get_quantiles_with_weights(SR_stats_4b, weights_4b, w_4b_cut[::-1])
            w_signal = [np.sum(weights_signal[SR_stats_signal > SR_stats_thrs[i]]) / w_signal_sum for i in range(len(SR_stats_thrs))]
            
            for i in range(len(SR_stats_thrs)):
                df_list.append({
                    "w_signal": w_signal[i],
                    "w_4b_cut": w_4b_cut[i],
                    "seed": seed,
                    "signal_ratio": signal_ratio,
                    "n_3b": n_3b,
                    "noise_scale": noise_scale,
                })
                
df = pd.DataFrame(df_list)
df.to_csv("./data/tsv/SR_stats_vs_w_4b.csv", index=False, mode="a", header=False)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [03:41<00:00, 22.13s/it]
